# Netflix Natural Language Processing (NLP) Recommendation System

### Package Install
<p> All the packages that we'll need to use. Currently I've only added the ones I've need thus far, feel free to add any you think are necessary! </p>

In [19]:
!pip install nltk --quiet

In [21]:
import re # for removing punctuation
import urllib.request # for downloading the file if not present
import math # for log in IDF computation
import collections # for counting word frequencies

import numpy as np
import nltk
nltk.download('stopwords')
import pandas as pd

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jfleege15/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#### Dataset Info

In [38]:
# the datacamp dataset i think
netflix_t1_df = pd.read_csv('https://raw.githubusercontent.com/azaan-f/netflix-nlp-recommender/main/datasets/netflix_titles.csv')
netflix_t1_df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,81145628,Movie,Norm of the North: King Sized Adventure,"Richard Finn, Tim Maltby","Alan Marriott, Andrew Toth, Brian Dobson, Cole...","United States, India, South Korea, China","September 9, 2019",2019,TV-PG,90 min,"Children & Family Movies, Comedies",Before planning an awesome wedding for his gra...
1,80117401,Movie,Jandino: Whatever it Takes,NaN,Jandino Asporaat,United Kingdom,"September 9, 2016",2016,TV-MA,94 min,Stand-Up Comedy,Jandino Asporaat riffs on the challenges of ra...
2,70234439,TV Show,Transformers Prime,NaN,"Peter Cullen, Sumalee Montano, Frank Welker, J...",United States,"September 8, 2018",2013,TV-Y7-FV,1 Season,Kids' TV,"With the help of three human allies, the Autob..."
3,80058654,TV Show,Transformers: Robots in Disguise,NaN,"Will Friedle, Darren Criss, Constance Zimmer, ...",United States,"September 8, 2018",2016,TV-Y7,1 Season,Kids' TV,When a prison ship crash unleashes hundreds of...
4,80125979,Movie,#realityhigh,Fernando Lebrija,"Nesta Cooper, Kate Walsh, John Michael Higgins...",United States,"September 8, 2017",2017,TV-14,99 min,Comedies,When nerdy high schooler Dani finally attracts...


In [60]:
netflix_t1_df.isnull().any()

show_id         False
type            False
title           False
director         True
cast             True
country          True
date_added       True
release_year    False
rating           True
duration        False
listed_in       False
description     False
dtype: bool

In [40]:
# larger of the two i believe
netflix_t2_df = pd.read_csv('https://raw.githubusercontent.com/azaan-f/netflix-nlp-recommender/main/datasets/titles.csv')
netflix_t2_df.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,48,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,113,"['crime', 'drama']",['US'],NaN,tt0075314,8.3,795222.0,27.612,8.2
2,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['comedy', 'fantasy']",['GB'],NaN,tt0071853,8.2,530877.0,18.216,7.8
3,tm70993,Life of Brian,MOVIE,"Brian Cohen is an average young Jewish man, bu...",1979,R,94,['comedy'],['GB'],NaN,tt0079470,8.0,392419.0,17.505,7.8
4,tm190788,The Exorcist,MOVIE,12-year-old Regan MacNeil begins to adapt an e...,1973,R,133,['horror'],['US'],NaN,tt0070047,8.1,391942.0,95.337,7.7


In [55]:
netflix_t2_df.isnull().any()

id                      False
title                    True
type                    False
description              True
release_year            False
age_certification        True
runtime                 False
genres                  False
production_countries    False
seasons                  True
imdb_id                  True
imdb_score               True
imdb_votes               True
tmdb_popularity          True
tmdb_score               True
dtype: bool

In [49]:
u = 'https://raw.githubusercontent.com/azaan-f/netflix-nlp-recommender/main/datasets/users.csv'

netflix_user_df = pd.read_csv(u, encoding="latin1")
netflix_user_df.head()

,UserID,FavoriteGenres,WatchedMovies,EvaluationMoviesTheyWillLike,AvgDescriptionReadTimeSeconds,ProfileType
0,U01,"Action, Crime, Thriller","Ronny Chieng: Speakeasy, A Cinderella Story, H...","Power Rangers Lost Galaxy, Puppy Star Christma...",14,Action/Crime
1,U02,"Action, Sci-Fi","No Good Nick, Rica, Famosa, Latina, E-Team, Co...","Scary Movie 5, Jack Whitehall: Christmas with ...",55,Action/Sci-Fi
2,U03,"Comedy, Romance","Thalaivii, Monty Python: The Meaning of Live, ...","Schitt's Creek, Undefeated, The Quick and the ...",20,Comedy/Romance
3,U04,"Drama, Romance, Mystery","Dj Vu, Love in a Puff, Elevator Baby, Surviv...","Killer Cove, Casting JonBenet, Hard Kill",19,Drama/Romance
4,U05,"Sci-Fi, Thriller","Myths & Monsters, The Burial of Kojo, Macchli ...","El Mago Pop, Raees, My Girl",38,Sci-Fi/Thriller


### Dataset Download Pre-Processing Logic
<p> This is the start of the pre-processing phase. This is where we'll first grab the datasets from the dataset folder. Here all of the self punctuations and stop words will be defined. </p>

<p> **keeping the "self" notation for now if we decide to move to a class, can always delete later</p>

In [29]:
def __init__(self):
    # csvs from the dataset folder
    title_url = 'https://raw.githubusercontent.com/azaan-f/netflix-nlp-recommender/main/datasets/netflix_titles.csv'
    title_url2 = 'https://raw.githubusercontent.com/azaan-f/netflix-nlp-recommender/main/datasets/titles.csv' # larger of the two i believe
    users = 'https://raw.githubusercontent.com/azaan-f/netflix-nlp-recommender/main/datasets/users.csv'

    titles_df = pd.read_csv(title_url) # swap if necessary

    # punctuation/stop word logic
    self.punctuations = '"\,<>./?@#$%^&*_~/!()-[]{};:'

    try:
        _ = stopwords.words('english')
    except LookupError:
        nltk.download('stopwords', quiet=True)

    self.stop_words = set(stopwords.words('english'))

    self.dataset = titles_df

### Normalizer
<p> Logic for all text normalization (stripped, lowered, etc.) will go here.</p>

In [35]:
# normalizer
def normalize(self, text):
    text = "" if text is None else str(text)
    text = text.strip().lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [t for t in tokens if t not in self.stop_words]

    return " ".join(tokens)

In [34]:
def preprocess_descriptions(self, description_col="description"):
    self.dataset[description_col] = (
        self.dataset[description_col]
        .fillna("")
        .astype(str)
        .apply(self.normalize)
    )
    return self.dataset

### text2Bit and Bit Vector Score
<p> Return the bit vector representation of the text. </p>

In [71]:
def build_vocabulary(self, text_col="description", max_words=200):
    if self.dataset is None:
      raise Exception("data set not loaded")
    
    # count the ocurrance of the words in the dataset 
    count = collections.Counter()

    for doc in self.dataset[text_col].fillna("").astype(str):
        count.update(doc.split())

    most_common = count.most_common(max_words)
    words_only = [w for (w, c) in most_common]

    self.vocab = np.array(words_only, dtype=object)

In [62]:
def text2BitVector(self,text):
    bitVector = np.zeros(self.vocab.size, dtype=np.int32)
    tokens = set(self.normalize(text).split())

    for i, w in enumerate(self.vocab):
      if w in tokens:
        bitVector[i] = 1 # if present, set to 1, otherwise it remains 0

    return bitVector

In [47]:
def bit_vector_score(self, query,doc):
    q = self.text2BitVector(query)
    d = self.text2BitVector(doc)

    # compute the relevance using q and d
    relevance = float(q.dot(d))
    return relevance

In [64]:
# executes the computation of the relevance score for each document
def execute_search_BitVec(self, query, text_col="description"):
    query = self.normalize(query)

    relevances = np.zeros(self.dataset.shape[0])

    for i, doc in enumerate(self.dataset[text_col].fillna("").astype(str)):
        relevances[i] = self.bit_vector_score(query, doc)

    return relevances

### TF-IDF

In [67]:
def compute_IDF(self,M,collection):
    self.IDF = np.zeros(self.vocab.size) # initialize the IDFs to zero
    doc_sets = [set(self.normalize(doc).split()) for doc in collection.fillna("").astype(str)]

    for i, w in enumerate(self.vocab):
        df = sum(1 for s in doc_sets if w in s)
        self.IDF[i] = math.log((M + 1) / (df + 1)) + 1 # formula from slides

In [68]:
# returns the bit vector representation of the text
def text2TFIDF(self,text, applyBM25_and_IDF=False):
    tokens = self.normalize(text).split()
    tfidfVector = np.zeros(self.vocab.size, dtype=float)

    if not tokens:
        return tfidfVector

    counts = collections.Counter(tokens) 
    
    for i, w in enumerate(self.vocab):
        tf = counts.get(w, 0)
        if tf > 0:
            tfidfVector[i] = tf * self.IDF[i]

    return tfidfVector

In [69]:
def tfidf_score(self,query,doc, applyBM25_and_IDF=False):
    q = self.text2TFIDF(query)
    d = self.text2TFIDF(doc)

    relevance = float(q.dot(d))
    return relevance

In [72]:
def execute_search_TF_IDF(self,query):
    query = self.normalize(query)
    self.compute_IDF(self.dataset.shape[0], self.dataset[text_col])

    # global IDF
    relevances = np.zeros(self.dataset.shape[0])

    for i, doc in enumerate(self.dataset[text_col].fillna("").astype(str)):
        relevances[i] = self.tfidf_score(query, doc)

    return relevances # in the same order of the documents in the dataset